In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
campaigns_bronze = '/Volumes/main/lakehouse_marketing/bronze/campaigns/'


df_campaigns = spark.read\
                    .format("delta")\
                    .option("header", "true")\
                    .load(campaigns_bronze)

display(df_campaigns)
display(df_campaigns.printSchema())

In [0]:
df = df_campaigns.select(
    'campaign_id',
    'campaign_name',
    'channel',
    'start_date',
    'end_date',
    'ingestion_timestamp',
    'source_file'
)


df_typed = df\
            .withColumn('campaign_name', F.trim(F.col('campaign_name')))\
            .withColumn('start_date', F.to_date('start_date'))\
            .withColumn('end_date', F.to_date('end_date'))

df_typed.printSchema()

In [0]:
display(df_campaigns.groupBy("channel").count())

* Normalization

  * `Redefinir a regra para os canais no documento.` 

In [0]:
df_normalized = df_typed.withColumn(
    "channel",
    F.when(F.lower(F.regexp_replace("channel", "-", "")) == "email", "EMAIL")
    .when(F.lower(F.col("channel")).isin("social"), "SOCIAL")
    .otherwise("OTHER")
)

display(df_normalized.groupBy('channel').count())

#### Remove `campaign_id` nulos

In [0]:
df_normalized.printSchema()

In [0]:
df_with_rules = df_normalized.withColumn(
    "rejection_reason",
    F.when(F.col('campaign_id').isNull(), 'NULL_CAMPAIGN_ID')
     .when(F.col('channel').isNull(), 'NULL_CHANNEL')
     .otherwise(None)
)

df_with_rules.show()

In [0]:
# Regra 1: campaign_id não pode ser nulo
df_valid = df_with_rules.filter(F.col('rejection_reason').isNull())

# Regra 2: start_date <= end_date
df_valid = df_valid.filter(F.col("start_date") <= F.col("end_date"))

# Dados que não se enquadram na regra do tratamento
df_rejected = df_with_rules.filter(F.col('rejection_reason').isNotNull())

* Deduplicação

---
Aplica a deduplicação em `campaign_id` onde com a data da ingestão mais atual.

In [0]:
window = Window.partitionBy("campaign_id").orderBy(F.col("ingestion_timestamp").desc())

df_dedup = (
    df_valid
    .withColumn("row_number", F.row_number().over(window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print(f"Sem deduplicação: {df_valid.count()}")
print(f"Com deduplicação: {df_dedup.count()}")

#### Escrita na SILVER

In [0]:
################################################
######## ESCRITA DOS DADOS TRATADOS ############
################################################
df_valid.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable('main.silver_marketing.campaigns')

################################################
####### ESCRITA DOS DADOS SEM TRATAMENTO #######
################################################
df_rejected.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable('main.governance_marketing.campaigns_rejected')

* Verificações na tabela

In [0]:
%sql
SELECT 
    COUNT(*) AS n_registros
FROM 
    main.silver_marketing.campaigns

In [0]:
%sql
SELECT
    COUNT(*) AS n_registros
FROM main.governance_marketing.campaigns_rejected